# Operation ShadowVault — Ransomware Incident Response Analysis

**Scenario:** A manufacturing company (Meridian Precision Mfg) detects unusual
activity on a Windows endpoint. Employees report slow systems and missing
files. This notebook walks through the SOC investigation of a multi-stage
attack: **Initial Access → Credential Theft → Lateral Movement → Data
Exfiltration Attempt → Ransomware Deployment**.

The underlying data is a synthetic log dataset (`data/raw/`) generated by
`src/log_generator.py`, combining normal business-day noise with an
embedded attack chain. The detection logic in `src/detectors/` is applied
below stage by stage, then correlated into a single incident timeline.

> This notebook analyzes and detects simulated attacker *behavior* in log
> data — it does not contain or execute any functional malware, exploit,
> or encryption code.

In [ ]:
import sys
from pathlib import Path

project_candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_DIR = next(path for path in project_candidates if (path / "src" / "utils.py").exists())
sys.path.append(str(PROJECT_DIR / "src"))

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from utils import load_logs
from detectors import initial_access, credential_access, lateral_movement, exfiltration, ransomware
from correlation_engine import run_all_detectors, score_by_host, attack_chain_summary

pd.set_option("display.max_colwidth", 120)
sec, sysmon, fw, files = load_logs()

## 1. Raw Data Overview

Four log sources feed this investigation, mirroring what a real SOC
would pull from a SIEM: Windows Security events, Sysmon (EDR-style)
telemetry, perimeter firewall logs, and endpoint file-activity logs.

In [ ]:
for name, df in [("Windows Security", sec), ("Sysmon", sysmon),
                 ("Firewall", fw), ("File Activity", files)]:
    print(f"{name:18s} {len(df):>4} rows   columns: {list(df.columns)}")

In [ ]:
sysmon.head(5)

## 2. Stage-by-Stage Detection

Each detector below is intentionally narrow — it looks for one specific,
well-known attacker behavior (a technique, in ATT&CK terms) rather than
trying to classify "suspicious" activity broadly. That's what keeps the
false-positive rate low against the ~200 rows of benign noise mixed into
the dataset.

### Stage 1 — Initial Access (T1566.001 / T1204.002 / T1059.001)

In [ ]:
stage1 = pd.DataFrame(initial_access.detect(sysmon))
stage1

### Stage 2 — Credential Theft (T1003.001)

In [ ]:
stage2 = pd.DataFrame(credential_access.detect(sysmon, files))
stage2

### Stage 3 — Lateral Movement (T1021.002 / T1569.002)

In [ ]:
stage3 = pd.DataFrame(lateral_movement.detect(sec, sysmon))
stage3

### Stage 4 — Data Exfiltration Attempt (T1560 / T1041)

In [ ]:
stage4 = pd.DataFrame(exfiltration.detect(sysmon, fw))
stage4

### Stage 5 — Ransomware Deployment (T1490 / T1486 / T1070.001)

In [ ]:
stage5 = pd.DataFrame(ransomware.detect(sysmon, sec, files))
stage5

## 3. Correlated Incident Timeline

`correlation_engine.py` merges every stage's alerts into one chronological
timeline and scores hosts by cumulative severity, so an analyst can
immediately see the full kill chain and which assets were hit hardest.

In [ ]:
timeline = run_all_detectors()
risk = score_by_host(timeline)
summary = attack_chain_summary(timeline)

print(f"{len(timeline)} correlated alerts across {timeline['Stage'].nunique()} stages")
summary

## 4. Visualizations

**Attack timeline** — every alert plotted chronologically, colored by stage.

In [ ]:
fig = px.scatter(
    timeline, x="Timestamp", y="Stage", color="Stage", symbol="Severity",
    hover_data=["Hostname", "Account", "Technique", "MITRE_ID"],
    title="Operation ShadowVault — Attack Chain Timeline",
    height=450,
)
fig.update_traces(marker=dict(size=13, line=dict(width=1, color="white")))
fig.update_layout(showlegend=True, xaxis_title="Time", yaxis_title="Attack Stage")
fig.show()

**Host risk ranking** — cumulative severity-weighted score per asset.

In [ ]:
fig = px.bar(
    risk, x="Hostname", y="RiskScore", color="RiskScore",
    color_continuous_scale="Reds", title="Host Risk Score (severity-weighted alert count)",
    height=400,
)
fig.show()

**Lateral movement path** — who the stolen admin account (`j.alvarez`) touched, and when.

In [ ]:
lat_events = sec[(sec["EventID"] == 4624) & (sec["LogonType"] == 3) &
                 (sec["Account"] == "j.alvarez")].sort_values("Timestamp")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=lat_events["Timestamp"], y=lat_events["Hostname"],
    mode="markers+lines+text", text=lat_events["Hostname"], textposition="top center",
    marker=dict(size=14, color="crimson"), line=dict(color="lightgrey", dash="dot"),
))
fig.update_layout(title="Lateral Movement Path — j.alvarez (stolen IT admin credentials)",
                   xaxis_title="Time", yaxis_title="Host", height=400)
fig.show()

**Exfiltration spike** — outbound bytes over time, highlighting the anomalous transfer.

In [ ]:
fw_sorted = fw.sort_values("Timestamp")
fig = px.bar(
    fw_sorted, x="Timestamp", y="BytesSent", color="Action",
    color_discrete_map={"Allow": "orange", "Blocked": "green"},
    title="Outbound Traffic Volume Over Time (flagged transfer highlighted)",
    height=400,
)
fig.show()

## 5. Generated Incident Report

`report_generator.py` renders the same correlated data into a formatted
Markdown incident report (`reports/incident_report.md`) — the kind of
artifact you'd actually hand to a manager or attach to a resume writeup.

In [ ]:
from report_generator import build_report
report_md = build_report(timeline, risk, summary)
print(report_md[:1500] + "\n...\n[full report continues in reports/incident_report.md]")

## Conclusion

This simulation walks through a complete ransomware kill chain — phishing,
credential theft, lateral movement, attempted exfiltration, and encryption
— purely through log correlation and MITRE ATT&CK-mapped detection logic,
with no manual "we know where the attack is" shortcuts baked into the
detectors. That structure (independent, technique-scoped detectors feeding
a correlation layer) mirrors how real detection engineering teams build
and reason about SOC content.

**Possible extensions:**
- Add a sixth detector for C2 beaconing (regular-interval outbound connections).
- Swap the severity-weighted risk score for a proper anomaly-detection model.
- Feed `data/processed/incident_timeline.csv` into a SIEM-style dashboard (e.g. Streamlit).